In [3]:
import tensorflow as tf
from keras.models import Sequential

import numpy as np
import pandas as pd
from sklearn.kernel_approximation import RBFSampler
from sklearn.linear_model import SGDClassifier
from sklearn.model_selection import train_test_split
from sklearn import svm
from sklearn.metrics import classification_report
from sklearn import metrics
from sklearn.linear_model import LogisticRegression
from sklearn.naive_bayes import GaussianNB
from sklearn.neighbors import KNeighborsClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import (precision_score, recall_score,f1_score, accuracy_score,mean_squared_error,mean_absolute_error)
from sklearn.ensemble import AdaBoostClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import Normalizer

%config IPCompleter.greedy=True
#import seaborn as sns
import matplotlib as matplot
import matplotlib.pyplot as plt
%matplotlib inline
from IPython.core.interactiveshell import InteractiveShell
InteractiveShell.ast_node_interactivity = "all"
import warnings
warnings.filterwarnings("ignore")
from keras.models import Model, load_model
from keras.layers import *
from keras.callbacks import ModelCheckpoint
from keras import regularizers
from sklearn.metrics import *
from sklearn.ensemble import RandomForestClassifier, ExtraTreesClassifier, VotingClassifier
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder,normalize


Using TensorFlow backend.


In [4]:
train = pd.read_csv('UNSW_NB15_training-set.csv')
test = pd.read_csv('UNSW_NB15_testing-set.csv')

In [5]:
# train = pd.DataFrame(train[train['attack_cat'] == "Worms"])
# test = pd.DataFrame(test[test['attack_cat'] == "Worms"])

In [6]:
combined_data = pd.concat([train, test]).drop(['id'],axis=1)

In [7]:
train.shape
test.shape
combined_data.shape
train.head(3)
combined_data.head(3)

(82332, 45)

(175341, 45)

(257673, 44)

,id,dur,proto,service,state,spkts,dpkts,sbytes,dbytes,rate,...,ct_dst_sport_ltm,ct_dst_src_ltm,is_ftp_login,ct_ftp_cmd,ct_flw_http_mthd,ct_src_ltm,ct_srv_dst,is_sm_ips_ports,attack_cat,label
0,1,0.000011,udp,-,INT,2,0,496,0,90909.0902,...,1,2,0,0,0,1,2,0,Normal,0
1,2,0.000008,udp,-,INT,2,0,1762,0,125000.0003,...,1,2,0,0,0,1,2,0,Normal,0
2,3,0.000005,udp,-,INT,2,0,1068,0,200000.0051,...,1,3,0,0,0,1,3,0,Normal,0


,dur,proto,service,state,spkts,dpkts,sbytes,dbytes,rate,sttl,...,ct_dst_sport_ltm,ct_dst_src_ltm,is_ftp_login,ct_ftp_cmd,ct_flw_http_mthd,ct_src_ltm,ct_srv_dst,is_sm_ips_ports,attack_cat,label
0,0.000011,udp,-,INT,2,0,496,0,90909.0902,254,...,1,2,0,0,0,1,2,0,Normal,0
1,0.000008,udp,-,INT,2,0,1762,0,125000.0003,254,...,1,2,0,0,0,1,2,0,Normal,0
2,0.000005,udp,-,INT,2,0,1068,0,200000.0051,254,...,1,3,0,0,0,1,3,0,Normal,0


In [8]:
# get_feature_names([input_features])

In [9]:
# train = train.drop(['id'],axis=1)
# test = test.drop(['id'],axis=1)

In [10]:
# Contaminsation mean pollution (outliers) in data
tmp = train.where(train['attack_cat'] == "Normal").dropna()
contamination = round(1 - len(tmp)/len(train), 2)
print("train contamination ", contamination)

tmp = test.where(test['attack_cat'] == "Normal").dropna()
print("test  contamination ", round(1 - len(tmp)/len(test),2),'\n')

if contamination > 0.5:
    print(f'contamination is {contamination}, which is greater than 0.5. Fixing...')
    contamination = round(1-contamination,2)
    print(f'contamination is now {contamination}')

train contamination  0.55
test  contamination  0.68 

contamination is 0.55, which is greater than 0.5. Fixing...
contamination is now 0.45


In [11]:
le1 = LabelEncoder()
le = LabelEncoder()

vector = combined_data['attack_cat']

print("attack cat:", set(list(vector))) # use print to make it print on single line 

combined_data['attack_cat'] = le1.fit_transform(vector)
combined_data['proto'] = le.fit_transform(combined_data['proto'])
combined_data['service'] = le.fit_transform(combined_data['service'])
combined_data['state'] = le.fit_transform(combined_data['state'])

vector = combined_data['attack_cat']
print('\nDescribing attack_type: ')
print("min", vector.min())
print("max", vector.max())
print("mode",vector.mode(), "Which is,", le1.inverse_transform(vector.mode()))
print("mode", len(np.where(vector.values==6)[0])/len(vector),"%")
combined_data['attack_cat'].max()

attack cat: {'Fuzzers', 'Normal', 'Analysis', 'Generic', 'Shellcode', 'Worms', 'Exploits', 'Backdoor', 'Reconnaissance', 'DoS'}

Describing attack_type: 
min 0
max 9
mode 0    6
dtype: int32 Which is, ['Normal']
mode 0.3609225646458884 %


9

In [12]:
# combined_data.loc[387,:]
# train.loc[387,:]
# test.loc[387,:]

In [13]:
## OMITTED: For statistical feature removal

lowSTD = list(combined_data.std().to_frame().nsmallest(6, columns=0).index)
# this is stupid. suppose a feature has a 1.0 (spearman or pearson) correlation, OR conditional probability, when not 0.... That a very useful feature  

lowCORR = list(combined_data.corr().abs().sort_values('attack_cat')['attack_cat'].nsmallest(3).index) # .where(lambda x: x < 0.005).dropna()
# This might be stupid. A Deep MLP (feed forward neural net) may see patterns

drop = set( lowCORR + lowSTD)
print(drop)
drop = {'ackdat', 'ct_ftp_cmd', 'djit', 'is_ftp_login', 'is_sm_ips_ports', 'response_body_len', 'sjit', 'synack', 'tcprtt'}
print(drop)
# print(f'Before {combined_data.shape}')
combined_data_reduced=combined_data.drop(drop,axis=1)
# print(f'After {combined_data.shape}')

{'is_ftp_login', 'is_sm_ips_ports', 'tcprtt', 'ct_ftp_cmd', 'djit', 'synack', 'sjit', 'response_body_len', 'ackdat'}
{'is_ftp_login', 'is_sm_ips_ports', 'tcprtt', 'ct_ftp_cmd', 'djit', 'synack', 'sjit', 'response_body_len', 'ackdat'}


In [14]:
#combined_data_reduced = combined_data_reduced[combined_data_reduced['attack_cat']==1]

In [15]:
combined_data_reduced.head()

,dur,proto,service,state,spkts,dpkts,sbytes,dbytes,rate,sttl,...,ct_state_ttl,ct_dst_ltm,ct_src_dport_ltm,ct_dst_sport_ltm,ct_dst_src_ltm,ct_flw_http_mthd,ct_src_ltm,ct_srv_dst,attack_cat,label
0,0.000011,119,0,5,2,0,496,0,90909.0902,254,...,2,1,1,1,2,0,1,2,6,0
1,0.000008,119,0,5,2,0,1762,0,125000.0003,254,...,2,1,1,1,2,0,1,2,6,0
2,0.000005,119,0,5,2,0,1068,0,200000.0051,254,...,2,1,1,1,3,0,1,3,6,0
3,0.000006,119,0,5,2,0,900,0,166666.6608,254,...,2,2,2,1,3,0,2,3,6,0
4,0.000010,119,0,5,2,0,2126,0,100000.0025,254,...,2,2,2,1,3,0,2,3,6,0


In [16]:

data_x = combined_data_reduced.drop(['attack_cat','label'], axis=1) # droped label
data_y = combined_data_reduced.loc[:,['label']]
# del combined_data # free mem
X_train, X_test, y_train, y_test = train_test_split(data_x, data_y, test_size=.20, random_state=42) # TODO

In [17]:
X_train.head()

,dur,proto,service,state,spkts,dpkts,sbytes,dbytes,rate,sttl,...,trans_depth,ct_srv_src,ct_state_ttl,ct_dst_ltm,ct_src_dport_ltm,ct_dst_sport_ltm,ct_dst_src_ltm,ct_flw_http_mthd,ct_src_ltm,ct_srv_dst
102468,0.210396,113,5,4,10,10,1000,4664,90.305896,62,...,1,1,1,1,1,1,1,1,1,1
64802,5.619431,113,9,4,138,38,162294,2612,31.141943,62,...,0,1,1,1,1,1,3,0,1,1
33634,0.000977,119,2,2,2,2,132,164,3070.624396,31,...,0,6,0,1,5,1,1,0,11,3
27874,0.037950,113,0,4,60,62,3614,50036,3188.405658,31,...,0,7,0,1,1,1,1,0,3,1
99000,0.767809,113,7,4,22,30,1040,18514,66.422771,62,...,0,1,1,1,1,1,1,0,1,1


In [27]:
print(combined_data_reduced.columns.values)
combined_data_reduced.shape

['dur' 'proto' 'service' 'state' 'spkts' 'dpkts' 'sbytes' 'dbytes' 'rate'
 'sttl' 'dttl' 'sload' 'dload' 'sloss' 'dloss' 'sinpkt' 'dinpkt' 'swin'
 'stcpb' 'dtcpb' 'dwin' 'smean' 'dmean' 'trans_depth' 'ct_srv_src'
 'ct_state_ttl' 'ct_dst_ltm' 'ct_src_dport_ltm' 'ct_dst_sport_ltm'
 'ct_dst_src_ltm' 'ct_flw_http_mthd' 'ct_src_ltm' 'ct_srv_dst' 'attack_cat'
 'label']


(257673, 35)

In [21]:
print(data_x.columns.values)
print(data_y.columns.values)
data_y.shape


['dur' 'proto' 'service' 'state' 'spkts' 'dpkts' 'sbytes' 'dbytes' 'rate'
 'sttl' 'dttl' 'sload' 'dload' 'sloss' 'dloss' 'sinpkt' 'dinpkt' 'swin'
 'stcpb' 'dtcpb' 'dwin' 'smean' 'dmean' 'trans_depth' 'ct_srv_src'
 'ct_state_ttl' 'ct_dst_ltm' 'ct_src_dport_ltm' 'ct_dst_sport_ltm'
 'ct_dst_src_ltm' 'ct_flw_http_mthd' 'ct_src_ltm' 'ct_srv_dst']
['label']


(257673, 1)

In [23]:
def gererated_preprocess(generated_data,lbl):
    """为GAN生成的数据加上attack_type"""
    df = generated_data
    columns = lbl[:-1]
    df["label"] = pd.Series(["1"]*len(df), index=df.index)

    return df

In [45]:
# from preprocess import preprocess

# processor = preprocess()
#加入生成数据的R2L类别二分类


read_generated_data = pd.read_csv('./output1/fake_examples.csv', sep=",", header=None)
#print(read_generated_data.head(3))
generated_data = gererated_preprocess(read_generated_data)

#train_add_BiR2L = train_Normal.append(generated_data)

#train_add_BiR2L = processor.merge_df(df_train, generated_data)
#train_add_BiR2L = np.concatenate(df_train, generated_data)

train_BiR2L = train_Normal.append(train_R2L)
train_add_BiR2L = train_BiR2L.append(generated_data)
X_train_add_R2L,y_train_add_R2L = processor.split_df(train_add_BiR2L)

test_BiR2L = test_Normal.append(test_R2L)
X_test_R2L,y_test_R2L = processor.split_df(test_BiR2L)


         0         1         2         3         4         5         6   \
0  0.048335  0.355938  0.444868  0.418741  0.014372  0.015424  0.014763   
1  0.017697  0.451038  0.625151  0.496012  0.016196  0.016004  0.015868   
2  0.004356  0.530903  0.713354  0.767666  0.019706  0.021016  0.020079   

         7         8         9     ...           32        33        34  \
0  0.014651  0.014838  0.014803    ...     0.053108  0.037163  0.072840   
1  0.016207  0.016318  0.016335    ...     0.032798  0.056042  0.072308   
2  0.020203  0.020513  0.020441    ...     0.029848  0.026167  0.430395   

         35        36        37        38        39        40        41  
0  0.982967 -0.003413  0.203192 -0.239296  0.075370 -0.025537  0.999708  
1  0.194676 -0.011479  0.990477  0.969838  0.118637  0.104010  0.999898  
2  0.830500  0.014606  0.121026 -0.126987 -0.004049 -0.043954  0.999287  

[3 rows x 42 columns]


AttributeError: 'preprocess' object has no attribute 'lbl'

In [31]:
def create_df(self, train, test):
    combined_data = pd.concat([train, test]).drop(['id'],axis=1)
    # Contaminsation mean pollution (outliers) in data
    tmp = train.where(train['attack_cat'] == "Normal").dropna()
    contamination = round(1 - len(tmp)/len(train), 2)
    print("train contamination ", contamination)

    tmp = test.where(test['attack_cat'] == "Normal").dropna()
    print("test  contamination ", round(1 - len(tmp)/len(test),2),'\n')

    if contamination > 0.5:
        print(f'contamination is {contamination}, which is greater than 0.5. Fixing...')
        contamination = round(1-contamination,2)
        print(f'contamination is now {contamination}')
        
    le1 = LabelEncoder()
    le = LabelEncoder()
    vector = combined_data['attack_cat']
    print("attack cat:", set(list(vector))) # use print to make it print on single line 
    combined_data['attack_cat'] = le1.fit_transform(vector)
    combined_data['proto'] = le.fit_transform(combined_data['proto'])
    combined_data['service'] = le.fit_transform(combined_data['service'])
    combined_data['state'] = le.fit_transform(combined_data['state'])
    
    print(le1.inverse_transform([0,1,2,3,4,5,6,7,8,9]))
    
    lowSTD = list(combined_data.std().to_frame().nsmallest(6, columns=0).index)
    lowCORR = list(combined_data.corr().abs().sort_values('attack_cat')['attack_cat'].nsmallest(3).index) 
    drop = set( lowCORR + lowSTD)
    
    combined_data_reduced=combined_data.drop(drop,axis=1)
    #combined_data_reduced = combined_data_reduced[combined_data_reduced['attack_cat']==6]
    #['Analysis' 'Backdoor' 'DoS' 'Exploits' 'Fuzzers' 'Generic' 'Normal' 'Reconnaissance' 'Shellcode' 'Worms']
    #[0,1,2,3,4,5,6,7,8,9]

    remain_Worms = [0,1,2,3,4,5,6,7,8] #174
    remain_Shellcode = [0,1,2,3,4,5,6,7,9]#1511
    remain_Backedoor = [0,2,3,4,5,6,7,8,9]#2329
    remain_Analysis = [1,2,3,4,5,6,7,8,9]#2677
    
    remain_Dos = [0,1,3,4,5,6,7,8,9]#16353
    remain_Exploits = [0,1,2,4,5,6,7,8,9]#44525
    remain_Fuzzers= [0,1,2,3,5,6,7,8,9]#24246
    remain_Generic = [0,1,2,3,4,6,7,8,9]#215481
    remain_Reconnaissance = [0,1,2,3,4,5,6,8,9]#13987
    
    remain_Normal = [0,1,2,3,4,5,7,8,9]#2218761
    
    Worms = combined_data_reduced[~combined_data_reduced['attack_cat'].isin(remain_Worms)] 
    Shellcodes = combined_data_reduced[~combined_data_reduced['attack_cat'].isin(remain_Shellcodes)] 
    Backdoor = combined_data_reduced[~combined_data_reduced['attack_cat'].isin(remain_Backdoor)] 
    Analysis = combined_data_reduced[~combined_data_reduced['attack_cat'].isin(remain_Analysis)] 
    
    Dos = combined_data_reduced[~combined_data_reduced['attack_cat'].isin(remain_Dos)] 
    Exploits = combined_data_reduced[~combined_data_reduced['attack_cat'].isin(remain_Exploits)] 
    Fuzzers = combined_data_reduced[~combined_data_reduced['attack_cat'].isin(remain_Fuzzers)] 
    Generic = combined_data_reduced[~combined_data_reduced['attack_cat'].isin(remain_Generic)] 
    Reconnaissance = combined_data_reduced[~combined_data_reduced['attack_cat'].isin(remain_Reconnaissance)] 
    
    Normal = combined_data_reduced[~combined_data_reduced['attack_cat'].isin(remain_Normal)] 
    return Worms,Shellcodes,Backdoor,Analysis,Normal,  Dos,Exploits,Fuzzers,Generic,Reconnaissance



In [ ]:
self.Worms = self.Worms.drop(['attack_cat','label'], axis=1)
self.Normal = self.Normal.drop(['attack_cat','label'], axis=1)
self.Normal_train,self.Normal_test = train_test_split(self.Normal, test_size=.20, random_state=42)
self.Worms_train,self.Worms_test = train_test_split(self.Worms, test_size=.20, random_state=42)